# AI Avatar Desk Demo on Google Colab

Run the **real-mode** pipeline (Kokoro TTS → MuseTalk lip-sync → MP4) on a free Colab GPU and expose it through a public Cloudflare quick-tunnel so your laptop or Webex Desk can reach it.

## Prerequisites

1. **Runtime → Change runtime type → Hardware accelerator: GPU (T4)** before running anything.
2. (Optional) A Google account with Drive enabled, so MuseTalk's ~2 GB of weights survive across sessions and you don't have to re-download.
3. The `ai-avatar-desk-demo` repo. Three ways to get it onto Colab — pick one in cell 3:
   - **Best**: push the repo to GitHub and `git clone` it.
   - Upload a zip of the repo via the Files panel.
   - Mount Google Drive and copy it from there.

## What this notebook does

| Step | Cell | What it does |
|---|---|---|
| 1 | Sanity check | Confirms you have a GPU and Python 3.10 |
| 2 | Mount Drive (optional) | Persists MuseTalk weights between sessions |
| 3 | Get the repo | git clone, file upload, or copy from Drive |
| 4 | Install backend Python deps | FastAPI, Kokoro, etc. |
| 5 | Install MuseTalk + weights | The slow step (~15 min the first time) |
| 6 | Provide an avatar input | Upload a 5–10 s face video, or use a placeholder |
| 7 | Build the SPA | `npm run build` so the same backend serves the UI |
| 8 | Launch backend + Cloudflare tunnel | Prints your public URL |

## What you cannot do here

- Run a long-lived production service — Colab kills idle sessions after ~90 min on the free tier.
- Use the same URL twice — the Cloudflare quick-tunnel URL changes every time you re-run cell 8.
- Skip the avatar input — MuseTalk needs a face video to lip-sync onto.

Total wall time on the first run: **~25–30 min** (15 of which is the MuseTalk install). Subsequent sessions with Drive caching: **~3 min**.

## 1. Sanity check

Confirms the runtime has a GPU and the right Python version. Re-run this cell after the runtime is ready.

In [ ]:
import sys, subprocess, shutil, os

print('Python:', sys.version.split()[0])
assert sys.version_info[:2] >= (3, 10), 'Python 3.10+ required (Colab default is 3.10)'

if shutil.which('nvidia-smi'):
    print('--- nvidia-smi ---')
    print(subprocess.check_output(['nvidia-smi']).decode())
else:
    raise RuntimeError(
        'No GPU detected. Open Runtime → Change runtime type → Hardware accelerator: GPU (T4) and re-run.'
    )

free_gb = shutil.disk_usage('/content').free / 2**30
print(f'Free disk on /content: {free_gb:.1f} GB (you need >= 15 GB)')
assert free_gb > 15, 'Not enough disk; restart runtime.'

# Make sure a few system tools we rely on later are present.
for tool in ('git', 'ffmpeg', 'curl'):
    if shutil.which(tool) is None:
        print(f'Installing {tool}...')
        subprocess.check_call(['apt-get', '-qq', 'install', '-y', tool])
print('System tools OK.')

## 2. (Optional) Mount Google Drive

If you mount your Drive, MuseTalk weights are stored there once and reused next time, saving ~2 GB of download per session.

Skip this cell if you'd rather start fresh each session.

In [ ]:
USE_DRIVE = True   # set to False to skip Drive caching

MUSETALK_WEIGHTS_CACHE = None
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    cache_root = '/content/drive/MyDrive/avatar-demo-cache'
    os.makedirs(cache_root, exist_ok=True)
    MUSETALK_WEIGHTS_CACHE = f'{cache_root}/musetalk_models'
    print('MuseTalk weights cache:', MUSETALK_WEIGHTS_CACHE)
else:
    print('Drive caching disabled; weights will be re-downloaded each session.')

## 3. Get the repo onto Colab

Pick **one** path. Default is git clone (replace the URL with your fork).

In [ ]:
REPO_DIR = '/content/ai-avatar-desk-demo'

# --- Option A: git clone (recommended) ---
GIT_URL = 'https://github.com/rajithaBK/ai-avatar-scratch.git'
GIT_BRANCH = 'main'

# --- Option B: zip upload (uncomment to use) ---
# from google.colab import files
# uploaded = files.upload()           # pick ai-avatar-desk-demo.zip from your laptop
# zip_path = next(iter(uploaded))
# subprocess.check_call(['unzip', '-q', '-o', zip_path, '-d', '/content'])

# --- Option C: copy from Drive (uncomment to use) ---
# subprocess.check_call(['cp', '-r', '/content/drive/MyDrive/ai-avatar-desk-demo', '/content/'])

if not os.path.isdir(REPO_DIR):
    subprocess.check_call(['git', 'clone', '-b', GIT_BRANCH, GIT_URL, REPO_DIR])

os.chdir(REPO_DIR)
print('Repo at', REPO_DIR)
print(subprocess.check_output(['git', '-C', REPO_DIR, 'log', '-1', '--oneline']).decode().strip())

## 4. Install backend Python deps (Kokoro, FastAPI, etc.)

We install into the system Python (Colab is throwaway, no point in a venv). Takes ~3 min on first run.

In [ ]:
# Pip-install backend deps from requirements.txt, but pull torch from the
# CUDA 11.8 wheel index (Colab T4 GPUs work great with cu118 wheels).
subprocess.check_call([
    'pip', 'install', '--quiet',
    'torch==2.0.1', 'torchvision==0.15.2', 'torchaudio==2.0.2',
    '--index-url', 'https://download.pytorch.org/whl/cu118',
])

# Filter out the torch-CPU pin from our requirements file (we just installed CUDA torch).
req_file = f'{REPO_DIR}/backend/requirements.txt'
with open(req_file) as f:
    requirements = [
        line.strip() for line in f
        if line.strip() and not line.strip().startswith('#')
        and not line.strip().startswith('torch')
        # pip-system-certs is Windows-only; harmless on Linux but skip.
        and not line.strip().startswith('pip-system-certs')
    ]
subprocess.check_call(['pip', 'install', '--quiet', *requirements])

import torch
print('torch:', torch.__version__, 'cuda:', torch.cuda.is_available())
assert torch.cuda.is_available(), 'CUDA not available — GPU runtime required'

## 5. Install MuseTalk and download model weights

**This is the slow step.** Expect ~15 minutes the first time:
- pip install MuseTalk's deps (~3 min)
- mim install mmcv/mmdet/mmpose (~5 min)
- download_weights.sh (~5–10 min, 2 GB)

If you mounted Drive in cell 2, weights are reused on subsequent sessions.

In [ ]:
MUSETALK_DIR = f'{REPO_DIR}/third_party/MuseTalk'
os.makedirs(f'{REPO_DIR}/third_party', exist_ok=True)

if not os.path.isdir(MUSETALK_DIR):
    subprocess.check_call([
        'git', 'clone', '--depth', '1',
        'https://github.com/TMElyralab/MuseTalk.git', MUSETALK_DIR,
    ])

# MuseTalk's Python deps. We do this inside the same Colab Python because
# the runtime is short-lived; for AWS we'd use a separate venv.
subprocess.check_call([
    'pip', 'install', '--quiet', '-r', f'{MUSETALK_DIR}/requirements.txt',
])
subprocess.check_call(['pip', 'install', '--quiet', '-U', 'openmim'])
for pkg in ('mmengine', 'mmcv==2.0.1', 'mmdet==3.1.0', 'mmpose==1.1.0'):
    subprocess.check_call(['mim', 'install', pkg])

# Cache or download weights. download_weights.sh writes into ./models
weights_path = f'{MUSETALK_DIR}/models'
marker = f'{weights_path}/musetalkV15/unet.pth'
if MUSETALK_WEIGHTS_CACHE and os.path.exists(f'{MUSETALK_WEIGHTS_CACHE}/musetalkV15/unet.pth'):
    print('Restoring weights from Drive cache...')
    subprocess.check_call(['rsync', '-a', f'{MUSETALK_WEIGHTS_CACHE}/', f'{weights_path}/'])
elif not os.path.exists(marker):
    print('Downloading MuseTalk weights (~2 GB) ...')
    subprocess.check_call(['bash', f'{MUSETALK_DIR}/download_weights.sh'], cwd=MUSETALK_DIR)
    if MUSETALK_WEIGHTS_CACHE:
        os.makedirs(MUSETALK_WEIGHTS_CACHE, exist_ok=True)
        print('Caching weights to Drive for next session...')
        subprocess.check_call(['rsync', '-a', f'{weights_path}/', f'{MUSETALK_WEIGHTS_CACHE}/'])

assert os.path.exists(marker), 'MuseTalk weights still missing after install'
print('MuseTalk weights OK at', weights_path)

## 6. Avatar input

MuseTalk needs a 5–10 second face video to lip-sync onto. Pick one of the three options below.

Tips: person facing camera, neutral expression, mouth mostly closed when idle, professional/even lighting, plain background.

In [ ]:
AVATAR_DIR = f'{REPO_DIR}/assets/avatars'
AVATAR_PATH = f'{AVATAR_DIR}/default.mp4'
os.makedirs(AVATAR_DIR, exist_ok=True)

# --- Option A: upload from your laptop (recommended) ---
from google.colab import files
if not os.path.exists(AVATAR_PATH):
    print('Pick a 5–10 second face MP4 to upload as the avatar:')
    uploaded = files.upload()
    src = next(iter(uploaded))
    os.replace(src, AVATAR_PATH)

# --- Option B: download from a URL (uncomment to use) ---
# AVATAR_URL = 'https://example.com/face.mp4'
# subprocess.check_call(['curl', '-fsSL', AVATAR_URL, '-o', AVATAR_PATH])

# --- Option C: still-image placeholder (only if you really have nothing) ---
# Generates a 5 s loop of a still 'PLACEHOLDER' image so MuseTalk has
# something to run. The output WON'T look like a real avatar.
# subprocess.check_call(['ffmpeg', '-y', '-loglevel', 'error',
#     '-f', 'lavfi', '-i', 'color=c=0x1d2033:s=512x512:d=5',
#     '-vf', "drawtext=text='PLACEHOLDER':fontcolor=white:fontsize=42:x=(w-text_w)/2:y=(h-text_h)/2",
#     '-c:v', 'libx264', '-pix_fmt', 'yuv420p', AVATAR_PATH])

size_mb = os.path.getsize(AVATAR_PATH) / 2**20
print(f'Avatar: {AVATAR_PATH} ({size_mb:.1f} MB)')

## 7. Build the SPA

We compile the React frontend so the FastAPI backend can serve it from `/`. That way one tunnel URL covers UI + API + MP4s.

In [ ]:
subprocess.check_call(['npm', 'ci'], cwd=f'{REPO_DIR}/frontend')
# Same-origin, no separate VITE_BACKEND_URL needed.
with open(f'{REPO_DIR}/frontend/.env.production', 'w') as f:
    f.write('VITE_BACKEND_URL=\n')
subprocess.check_call(['npm', 'run', 'build'], cwd=f'{REPO_DIR}/frontend')
print('SPA built at', f'{REPO_DIR}/frontend/dist')

## 8. Launch the backend and a Cloudflare tunnel

Starts uvicorn in the background and a Cloudflare quick-tunnel that exposes it to the public internet at a `*.trycloudflare.com` URL. **Re-run this cell to get a fresh tunnel** if the previous one was reclaimed.

In [ ]:
import subprocess, os, time, re, threading, signal, atexit, requests

# 1) Install cloudflared if missing.
if shutil.which('cloudflared') is None:
    print('Installing cloudflared...')
    subprocess.check_call([
        'wget', '-q',
        'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb',
        '-O', '/tmp/cloudflared.deb',
    ])
    subprocess.check_call(['dpkg', '-i', '/tmp/cloudflared.deb'])

# 2) Configure backend env to point at the cloned MuseTalk + Colab paths.
env_path = f'{REPO_DIR}/.env'
with open(env_path, 'w') as f:
    f.write(f'''APP_MODE=real
BACKEND_HOST=127.0.0.1
BACKEND_PORT=8000
ASSETS_DIR={REPO_DIR}/assets
AUDIO_DIR={REPO_DIR}/assets/audio
OUTPUT_DIR={REPO_DIR}/assets/outputs
AVATAR_INPUT={AVATAR_PATH}
KOKORO_VOICE=af_heart
KOKORO_LANG_CODE=a
MUSETALK_DIR={MUSETALK_DIR}
MUSETALK_INFERENCE_SCRIPT={MUSETALK_DIR}/scripts/inference.py
MUSETALK_CHECKPOINT_DIR={MUSETALK_DIR}/models
MUSETALK_PYTHON={shutil.which('python')}
MOCK_VIDEO_PATH={REPO_DIR}/assets/mock/mock_avatar.mp4
FRONTEND_DIST_DIR={REPO_DIR}/frontend/dist
''')

# 3) Stop any backend / tunnel from a previous run of this cell.
for label in ('uvicorn_proc', 'tunnel_proc'):
    p = globals().get(label)
    if p and p.poll() is None:
        try:
            p.terminate(); p.wait(timeout=5)
        except Exception:
            p.kill()

# 4) Start the backend.
log_path = '/tmp/uvicorn.log'
log_file = open(log_path, 'w')
uvicorn_proc = subprocess.Popen(
    ['python', '-m', 'uvicorn', 'app.main:app', '--host', '127.0.0.1', '--port', '8000', '--env-file', env_path],
    cwd=f'{REPO_DIR}/backend',
    stdout=log_file, stderr=log_file,
    env={**os.environ, 'PYTHONPATH': f'{REPO_DIR}/backend'},
)
atexit.register(lambda: uvicorn_proc.terminate())

# Wait for the backend health endpoint.
ok = False
for i in range(60):
    try:
        if requests.get('http://127.0.0.1:8000/api/health', timeout=2).status_code == 200:
            ok = True
            break
    except Exception:
        pass
    time.sleep(1)
if not ok:
    print(open(log_path).read()[-4000:])
    raise RuntimeError('Backend failed to start — see log above.')
print('Backend up at http://127.0.0.1:8000')

# 5) Start a Cloudflare quick-tunnel and parse out the URL.
tunnel_log = '/tmp/cloudflared.log'
tlog_file = open(tunnel_log, 'w')
tunnel_proc = subprocess.Popen(
    ['cloudflared', 'tunnel', '--url', 'http://127.0.0.1:8000', '--no-autoupdate'],
    stdout=tlog_file, stderr=subprocess.STDOUT,
)
atexit.register(lambda: tunnel_proc.terminate())

tunnel_url = None
for _ in range(60):
    if os.path.exists(tunnel_log):
        with open(tunnel_log) as f:
            for line in f:
                m = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', line)
                if m:
                    tunnel_url = m.group(0); break
    if tunnel_url:
        break
    time.sleep(1)

if not tunnel_url:
    print(open(tunnel_log).read()[-4000:])
    raise RuntimeError('Cloudflare tunnel did not come up')

print('\n' + '=' * 60)
print('PUBLIC URL :', tunnel_url)
print('UI         :', tunnel_url + '/')
print('Health API :', tunnel_url + '/api/health')
print('=' * 60)
print('\nKeep this Colab tab open. Closing it stops the backend + tunnel.')
print('\nTo see backend logs run:    !tail -f /tmp/uvicorn.log')
print('To see tunnel  logs run:    !tail -f /tmp/cloudflared.log')

## How to use it

1. **Click the public URL printed by cell 8** — it serves the same React UI you'd see locally, with one difference: API calls go to the same origin so there's no CORS / proxy config to do.
2. Type a sentence, hit **Generate Avatar Video**.
3. The first job is slow because MuseTalk has to extract landmarks from your avatar (~30–60 s on a T4). Subsequent jobs reuse the cache and are 2–3× faster.
4. To use this URL on a Webex Desk, paste it as a Web App URL in Cisco Control Hub.

## Troubleshooting

- **`No GPU detected`**: Runtime → Change runtime type → GPU → Save → Connect.
- **Backend log says `MuseTalk failed`**: run `!tail -200 /tmp/uvicorn.log` and look at the captured stdout/stderr from the inference subprocess. Almost always a missing weight file or an out-of-memory on a particularly long input.
- **Tunnel URL changes every session**: that's a property of the free Cloudflare quick-tunnel. For a stable URL, sign up for a free Cloudflare account and use a named tunnel (`cloudflared tunnel create avatar-demo`). I left the simpler quick-tunnel here on purpose so this notebook needs zero auth.
- **Colab idle-disconnect**: the free runtime kicks you off after ~90 min idle. Re-run cell 8 (and only cell 8) on the next session — the deps from cells 4–7 stay in `/content`.

## When you're done

Just close the Colab tab. There's nothing to clean up; everything is in the runtime VM. Drive cache (if you enabled it) keeps MuseTalk weights ready for next time.